<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/day02-discussion-2.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Day 2 — In-class discussion problem (2 of 3)

Work this out **by hand in your group first** — then run the code cell
to check your answer before presenting.

## Does calling .backward() twice give you 2x the gradient, or something else?

```python
x = torch.tensor(3.0, requires_grad=True)
y = x**2
y.backward()
print(x.grad)          # first call

y2 = x**2
y2.backward()
print(x.grad)          # second call -- note: no zero_grad() in between
```

**As a group:**

1. By hand: what is dy/dx at x=3 for y = x²? What should `x.grad` print
   after the first `backward()` call?
2. Predict what the second `print(x.grad)` shows. Is it the same value
   again, or something different?
3. If it's different, what arithmetic operation produced that number?

Run the cell below to check your answers.

In [1]:
import torch

x = torch.tensor(3.0, requires_grad=True)
y = x**2
y.backward()
print("grad after 1st backward():", x.grad.item())

y2 = x**2
y2.backward()
print("grad after 2nd backward() (no zero_grad in between):", x.grad.item())

grad after 1st backward(): 6.0
grad after 2nd backward() (no zero_grad in between): 12.0


/home/arnee/miniforge3/envs/ekman-teaching/lib/python3.11/site-packages/torch/autograd/graph.py:825: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 804: forward compatibility was attempted on non supported HW (Triggered internally at /opt/conda/conda-bld/pytorch_1729647429097/work/c10/cuda/CUDAFunctions.cpp:108.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


**Discussion point:** dy/dx = 2x = 6 at x=3, so the first
`backward()` correctly sets `x.grad` to 6. But PyTorch **accumulates**
gradients into `.grad` by default rather than replacing them — the
second `backward()` computes another 6 and *adds* it to what's already
there, giving 12, not 6. This is exactly why every training loop you'll
see from here on calls something like `optimizer.zero_grad()` (or
`w.grad.zero_()`, as Day 6's notebook does directly) at the start of
every iteration — without it, gradients from previous steps silently
keep piling up.